# Week 1: Python & NumPy Mastery
## Notebook 11: Sparse Matrix Formats (COO, CSR, CSC) & SpMM from Scratch

### 1. Conceptual Intuition & Theoretical Background
In natural language processing (TF-IDF matrices), graph neural networks (adjacency graphs), and recommender systems (user-item matrices), datasets often contain over $99.5\%$ zeros. Storing these as dense $N \times M$ float arrays wastes vast RAM:
$$\text{Dense Memory} = N \times M \times 8 \text{ bytes}$$

**Sparse Storage Formats:**
1. **Coordinate Format (COO):**
   Stores triplets $(row, col, value)$. Ideal for incrementally building matrices:
   $$\text{Entries} = (i, j, v)$$
2. **Compressed Sparse Row (CSR):**
   Compacts row indices into row pointers (`indptr`) of size $N + 1$. Enables $O(1)$ row slicing and high-speed Sparse-Dense Matrix Multiplication (SpMM):
   $$\text{Memory}_{	ext{CSR}} = (2 \cdot \text{nnz} + N + 1) \times 8 \text{ bytes}$$
3. **Compressed Sparse Column (CSC):**
   Compacts column indices into column pointers. Ideal for efficient column slicing and operations like sparse PCA.

---

### 2. Scratch Python Implementation: CSR Matrix & SpMM Kernel

In [ ]:
import numpy as np

class CSRMatrixScratch:
    def __init__(self, data, indices, indptr, shape):
        self.data = np.array(data, dtype=np.float64)
        self.indices = np.array(indices, dtype=np.int32)
        self.indptr = np.array(indptr, dtype=np.int32)
        self.shape = shape
        self.nnz = len(self.data)

    @classmethod
    def from_dense(cls, dense_matrix):
        dense = np.asarray(dense_matrix)
        rows, cols = dense.shape
        data = []
        indices = []
        indptr = [0]
        
        for r in range(rows):
            non_zeros = np.nonzero(dense[r])[0]
            for c in non_zeros:
                data.append(dense[r, c])
                indices.append(c)
            indptr.append(len(data))
            
        return cls(data, indices, indptr, (rows, cols))

    def spmm(self, dense_vector):
        """Sparse Matrix-Dense Vector Multiplication: y = A @ x"""
        rows, cols = self.shape
        x = np.asarray(dense_vector).ravel()
        y = np.zeros(rows, dtype=np.float64)
        
        for r in range(rows):
            start = self.indptr[r]
            end = self.indptr[r + 1]
            # Dot product over only non-zero entries in row r
            for idx in range(start, end):
                c = self.indices[idx]
                y[r] += self.data[idx] * x[c]
                
        return y

# Benchmark: 1000 x 1000 matrix with 99% sparsity
np.random.seed(42)
dim = 500
dense = np.zeros((dim, dim))
nnz_count = int(dim * dim * 0.01) # 1% non-zeros
rand_r = np.random.randint(0, dim, nnz_count)
rand_c = np.random.randint(0, dim, nnz_count)
dense[rand_r, rand_c] = np.random.randn(nnz_count)

csr = CSRMatrixScratch.from_dense(dense)
x_vec = np.random.randn(dim)

y_scratch = csr.spmm(x_vec)
y_dense = np.dot(dense, x_vec)

dense_bytes = dense.nbytes
csr_bytes = csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes

print("--- Sparse CSR Compression & SpMM ---")
print(f"Dense Matrix RAM Footprint: {dense_bytes / 1024:.1f} KB")
print(f"CSR Matrix RAM Footprint:   {csr_bytes / 1024:.1f} KB")
print(f"RAM Compression Ratio:      {(1 - csr_bytes / dense_bytes) * 100:.1f}%")
print(f"SpMM Verification Max Diff: {np.max(np.abs(y_scratch - y_dense)):.1e}")

### 3. Step-by-Step Code Walkthrough
1. **`indptr` Encoding:** An array of size $\text{rows} + 1$. Row $i$ entries are bounded by indices $[	ext{indptr}[i], 	ext{indptr}[i+1])$.
2. **SpMM Calculation:** Skips all zero entries entirely, achieving $O(\text{nnz})$ runtime instead of $O(N \times M)$.
3. **Verification:** Matches dense matrix-vector product with floating-point precision.

---

### 4. Industry Applications & Trade-offs
- **Recommender Systems & Search Engines:** PageRank and Matrix Factorization on web adjacency matrices with billions of nodes.
- **Trade-off:** Modifying the sparsity pattern of a CSR matrix is expensive because inserting a new non-zero requires shifting arrays in memory.

---

### 5. Top Interview Questions & Answers
**Q1: Why is CSR preferred for row slicing while CSC is preferred for column slicing?**
> **Answer:** In CSR, all elements of row $i$ reside consecutively in the `data` array. In CSC, elements of column $j$ reside consecutively, enabling contiguous memory reads.

**Q2: What is the time complexity of adding two CSR sparse matrices?**
> **Answer:** $O(\text{nnz}_A + \text{nnz}_B)$. It works like merging two sorted arrays row by row using the column indices.